# Facility location lab: the p-median problem
Choose a fixed number of candidate service sites to minimise weighted demand-to-facility cost.

This notebook uses a small Euclidean example and exhaustive enumeration. It is an exact teaching baseline, not a city-scale solver.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'implementations').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'implementations/python/p-median'))
from p_median import solve_p_median


## 1. Define demand and candidate sites

Rows in the cost matrix represent demand locations. Columns represent possible facilities. The demand weights can represent population, expected visits or another justified measure of demand.

In [ ]:
demand_xy = np.array([[0, 0], [2, 1], [5, 0], [1, 5], [4, 4], [7, 5], [2, 8], [8, 8]])
demand_weights = np.array([2, 1, 3, 2, 4, 1, 2, 3], dtype=float)
candidate_xy = np.array([[0, 1], [3, 0], [6, 1], [0, 6], [4, 5], [8, 7]])
costs = np.linalg.norm(demand_xy[:, None, :] - candidate_xy[None, :, :], axis=2)
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(demand_xy[:, 0], demand_xy[:, 1], s=30 + 25 * demand_weights, label='Demand (size = weight)')
ax.scatter(candidate_xy[:, 0], candidate_xy[:, 1], marker='s', s=90, facecolors='none', edgecolors='black', label='Candidate facility')
ax.set_title('Demand locations and candidate sites'); ax.set_aspect('equal'); ax.legend(); ax.grid(True); plt.show()


## 2. Open exactly two facilities

The objective is the sum of demand weight multiplied by the cost to the nearest open facility. This is not the p-center objective, which minimises the maximum service cost.

In [ ]:
opened, objective, assignment = solve_p_median(costs, p=2, demand_weights=demand_weights)
print('Open candidate indices:', opened)
print('Weighted total cost:', round(objective, 3))
print('Assignments:', assignment)
fig, ax = plt.subplots(figsize=(7, 6))
for i, facility in enumerate(assignment):
    ax.plot([demand_xy[i, 0], candidate_xy[facility, 0]], [demand_xy[i, 1], candidate_xy[facility, 1]], alpha=.55)
ax.scatter(demand_xy[:, 0], demand_xy[:, 1], s=30 + 25 * demand_weights, label='Demand')
ax.scatter(candidate_xy[:, 0], candidate_xy[:, 1], marker='s', s=75, facecolors='none', edgecolors='grey', label='Candidate')
ax.scatter(candidate_xy[list(opened), 0], candidate_xy[list(opened), 1], marker='*', s=240, label='Selected facility')
ax.set_title('Nearest-open-facility assignment'); ax.set_aspect('equal'); ax.legend(); ax.grid(True); plt.show()


## 3. Change demand weights

A location with greater demand contributes more to the objective. Increase one weight and see whether the chosen facilities change.

In [ ]:
changed_weights = demand_weights.copy(); changed_weights[0] *= 10
changed = solve_p_median(costs, p=2, demand_weights=changed_weights)
print('Original facilities and objective:', opened, round(objective, 3))
print('Changed facilities and objective:', changed[0], round(changed[1], 3))


## 4. Investigate

1. Compare p=1, p=2 and p=3.
2. Replace Euclidean distance with a directed travel-time matrix. What changes?
3. Add a maximum service distance. Why does that change the optimisation model?
4. Explain why exhaustive enumeration becomes impractical as the number of candidate sites grows.
5. Which equity or capacity constraints would a real service-planning problem need?